<a href="https://colab.research.google.com/github/Jupeid/interactivebook/blob/main/TesteLivroInt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
class Personagem:
  def __init__(self,nome):
    self.nome = nome
    self.nivel = 0
    self.experiencia = 0
    self.exp_para_proximo_nivel = 10
    self.pontos_disponiveis = 0
    self.pode_distribuir_pontos = False
# ------ATRIBUTOS BASE------
    self.forca = 0
    self.destreza = 0
    self.inteligencia = 1
    self.sorte = 1
    self.magia = 0
    self.vitalidade = 0
# ------STATUS DERIVADOS------
    self.recalcular_status_maximos()
    self.hp_atual = self.hp_max
    self.mp_atual = self.mp_max

# -----COMPANIONS/INIMIGOS------
    self.npcs = {}
# -----INVENTARIO/MOEDAS------
    self.inventario = []
    self.equipamentos = {}
    self.moedas = 0
# -----METODOS DE NPCS------
  def registrar_npc(self, id_npc, npc_objeto=None, nome=None, faccao="neutro"):
    if id_npc not in self.npcs:
      if isinstance(npc_objeto, NPC):
        self.npcs[id_npc] = npc_objeto
      else:
        self.npcs[id_npc] = NPC(nome=nome, faccao=faccao)

      print(f"\n [MUNDO] Você conheceu: {self.npcs[id_npc].nome}!")

  def obter_npc(self, id_npc):
    return self.npcs.get(id_npc)

# -----METODOS DE INVENTARIO------
  def adicionar_item(self, item):
    if item not in self.inventario:
      self.inventario.append(item)
      print(f"\n [INVENTARIO] Você adquiriu: '{item}'!")

  def remover_item(self, item):
    if item in self.inventario:
      self.inventario.remove(item)
      print(f"\n [INVENTARIO] Você perdeu: '{item}'!")
      return True
    return False

# -----METODOS DE MOEDAS------
  def tem_moedas(self, quantidade):
    return self.moedas >= quantidade

  def adicionar_moedas(self, quantidade, multiplicador=1):
    valor_final = int(quantidade * multiplicador)
    self.moedas += valor_final
    print(f"\n [MOEDAS] Você ganhou {valor_final} moedas! Saldo atual {self.moedas}C")

  def gastar_moedas(self, quantidade):
    if self.tem_moedas(quantidade):
      self.moedas -= quantidade
      print(f"\n [MOEDAS] Você gastou {quantidade} moedas. Saldo atual {self.moedas}C")
      return True
    print(f"\n [!] Você não possui moedas suficientes!")
    return False

  def dividir_moedas(self, quantidade, divisao=2):
    valor_dividido = int(quantidade // divisao)
    self.moedas += valor_dividido
    print(f"\n [MOEDAS] Você recebeu {valor_dividido} moedas! Saldo atual {self.moedas}C")

# ------METODO DE TRAVA----
  def permitir_distribuicao(self, permitir:bool):
    self.pode_distribuir_pontos = permitir

# -----SISTEMA DE CONSULTA DE ATRIBUTOS COM BONUS------
  def obter_atributo_total(self, nome_atributo):
    nome = nome_atributo.lower()
    valor_base = getattr(self, nome_atributo, 0)
    bonus_equipamentos =  0
    for slot, item, in self.equipamentos.items():
      if isinstance(item, dict) and "bonus" in item:
        bonus_equipamentos += item["bonus"].get(nome_atributo, 0)
    return valor_base + bonus_equipamentos

  def recalcular_status_maximos(self):
    vitalidade_total = self.obter_atributo_total("vitalidade")
    magia_total = self.obter_atributo_total("magia")

    self.hp_max = 20 + (vitalidade_total * 5)
    self.mp_max = magia_total * 5

# ------TESTE ATRIBUTOS-----
  def tem_atributo(self, nome_atributo, valor_minimo):
    return self.obter_atributo_total(nome_atributo) >= valor_minimo

  def tem_mp(self, custo_mp):
    return self.mp_atual >= custo_mp

# -----ALTERA ATRIBUTOS-----
  def consumir_mp(self, quantidade):
    if self.tem_mp(quantidade):
      self.mp_atual -= quantidade
      print(f"\n [MP] Você gastou {quantidade} de MP. MP Atual: {self.mp_atual}/{self.mp_max}")
      return True
    return False

  def receber_dano(self, dano):
    self.hp_atual = max(0, self.hp_atual - dano)
    print(f"\n [DANO] Você sofreu {dano} de dano! HP: {self.hp_atual}/{self.hp_max}")

  def descansar(self):
    self.hp_atual = self.hp_max
    self.mp_atual = self.mp_max
    print("\n [DESCANSO] Seu HP e MP forma totalmente restaurados!")

# -----SISTEMA DE EQUIPAR E DESEQUIPAR ITENS-----
  def equipar_item(self, item_dict):
    slot = item_dict.get("slot")
    nome_item = item_dict.get("nome")

    if nome_item not in self.inventario:
      print(f"\n [!] Você não possui '{nome_item}' no inventário para equipar.")
      return False

    if slot in self.equipamentos:
      self.desequipar_item(slot)

    self.equipamentos[slot] = item_dict
    self.recalcular_status_maximos()
    print(f"\n [EQUIPAMENTO] Você equipou '{nome_item}' no slot [{slot.upper()}]!")
    return True

  def desequipar_item(self, slot):
    if slot in self.equipamentos:
      item_removido = self.equipamentos.pop(slot)
      self.recalcular_status_maximos()

      self.hp_atual = min(self.hp_atual, self.hp_max)
      self.mp_atual = min(self.mp_atual, self.mp_max)

      print(f"\n [EQUIPAMENTO] Você desequipou '{item_removido['nome']}'.")
    return None

# -----SISTEMA DE NIVEL E EXPERIENCIA-----
  def ganhar_xp(self, quantidade_xp):
    self.experiencia += quantidade_xp
    print(f"\n [XP] Você ganhou {quantidade_xp} de XP! ({self.experiencia}/{self.exp_para_proximo_nivel})")

    while self.experiencia >= self.exp_para_proximo_nivel:
        self.experiencia -= self.exp_para_proximo_nivel
        self.nivel += 1
        self.pontos_disponiveis += 1
        self.exp_para_proximo_nivel = int(self.exp_para_proximo_nivel * 1.5)
        print(f"\n NIVEL UP! Você subiu para o Nível {self.nivel}!")
        print(f"Você tem {self.pontos_disponiveis} ponto(s) de status para distribuir.")

  def distribuir_pontos(self, atributo):
    if not self.pode_distribuir_pontos:
      print("\n [!] Você só pode distribuir pontos em áreas de descanso"
      " ou trocas de capítulo!")
      return False

    if self.pontos_disponiveis <= 0:
      print("Você não possui pontos de status disponíveis!")
      return False

    atributo = atributo.lower()
    atributos_validos = [
        "forca",
        "destreza",
        "inteligencia",
        "sorte",
        "magia",
        "vitalidade",
    ]
    if atributo in atributos_validos and hasattr(self, atributo):
      valor_antigo = getattr(self, atributo)
      setattr(self, atributo, valor_antigo + 1)
      self.pontos_disponiveis -= 1

      self.recalcular_status_maximos()

      if atributo == "vitalidade":
        self.hp_atual += 5
      elif atributo == "magia":
        self.mp_atual += 5

      print(f"\n [STATUS] {atributo.capitalize()} aumentado para {getattr(self, atributo)}!")
      return True
    else:
      print(f"[!] Atributo inválido!")
      return False

In [ ]:
class Fragmento:
    def __init__(self, id_fragmento, nome, descricao, bonus: dict, slot="reliquia_1", portador_original=None):
        self.id_fragmento = id_fragmento
        self.nome = nome
        self.descricao = descricao
        self.bonus = bonus
        self.slot = slot
        self.portador_original = portador_original

    def to_dict(self):
        return {
            "id": self.id_fragmento,
            "nome": self.nome,
            "slot": self.slot,
            "bonus": self.bonus,
            "descricao": self.descricao
        }

    def exibir_detalhes(self):
        print(f"\n *[{self.nome.upper()}]*")
        print(f"Descrição: {self.descricao}")
        if self.portador_original:
            print(f"Origem/Portador: {self.portador_original}")
        print("Bônus Concedidos:")
        for attr, val in self.bonus.items():
            print(f" • +{val} em {attr.capitalize()}")

In [ ]:
# Banco de Dados dos Fragmentos
FRAGMENTOS_REGISTRADOS = {
    "frag_magia": Fragmento(
        id_fragmento="frag_magia",
        nome="Fragmento da Magia",
        descricao="Um pequeno cristal lapitado que brilha com energia arcana, o pulso de seu brilho parece alterar as leis da realidade ao seu redor.",
        bonus={"magia": 2, "forca": 1},
        portador_original="Astrid"
    ),
    "frag_guerra": Fragmento(
        id_fragmento="frag_guerra",
        nome="Fragmento da Guerra",
        descricao="Uma medalha militar que emite um calor sobrenatural, seu pulso ardente parece sedento por sangue.",
        bonus={"forca": 2, "vitalidade": 1},
        portador_original="Tyron"
    ),
    "frag_amor": Fragmento(
        id_fragmento="frag_amor",
        nome="Fragmento do Amor",
        descricao="Um brinco elegante, sua perola rubra parece encantar a todos que olhem diretamente para seu portador.",
        bonus={"destreza": 2, "magia": 1},
        portador_original="Serena"
    ),
    "frag_comercio": Fragmento(
        id_fragmento="frag_comercio",
        nome="Fragmento do Comercio",
        descricao="Um smartphone misterioso, dizem que é único no mundo, seu sistema parece revelar o preço de tudo... e todos.",
        bonus={"sorte": 2, "inteligencia": 1},
        portador_original="Siles"
    ),
    "frag_sabedoria": Fragmento(
        id_fragmento="frag_sabedoria",
        nome="Fragmento da Sabedoria",
        descricao="Um bloco de notas antigo, suas páginas atualizam sua escrita dourada frequentemente e seus textos parecem revelar todos os segredos do mundo.",
        bonus={"inteligencia": 2, "sorte": 1},
        portador_original="Soren"
    ),
        "frag_caca": Fragmento(
        id_fragmento="frag_caca",
        nome="Fragmento da Caça",
        descricao="Um anel de osso que exala uma aura primitiva, ao utilizar nada escapa dos sentidos de seu portador.",
        bonus={"destreza": 2, "forca": 1},
        portador_original="Naira"
    ),
    "frag_entretenimento": Fragmento(
        id_fragmento="frag_entretenimento",
        nome="Fragmento da entretenimento",
        descricao="Um dado de 6 lados colorido, muitos acreditam que ao possuir terá sorte imensa em qualquer jogo.",
        bonus={"sorte": 2, "vitalidade": 1},
        portador_original="Miles"
    ),
    "frag_terra": Fragmento(
        id_fragmento="frag_terra",
        nome="Fragmento da Terra",
        descricao="Uma semente dourada guardada dentro de um frasco de vidro, a semente parece exalar vitaliidade podendo crescer até no solo mais infértil.",
        bonus={"vitalidade": 2, "magia": 1},
        portador_original="Florence"
    ),
}

In [ ]:
# Banco de dados NPC Principais
PORTADORES_PRINCIPAIS = {
    "campeao_astrid": NPC(
        nome="Astrid",
        faccao="neutro",
        fragmento=FRAGMENTOS_REGISTRADOS["frag_magia"]
}

In [ ]:
class NPC:
    def __init__(self, nome, faccao="neutro", fragmento: Fragmento = None):
        self.nome = nome
        self.faccao = faccao #aliado, inimigo, neutro
        self.vivo = True
        self.fragmento = fragmento
        self.imobilizado = False

    def esta_disponivel(self):
        return self.vivo and not self.imobilizado

    def possui_fragmento(self):
        return self.fragmento is not None

    def entregar_fragmento(self):
        if self.fragmento:
            frag = self.fragmento
            self.fragmento = None
            print(f"\n *{self.nome} entregou o '{frag.nome}'!")
            return frag
        return None

    def abater(self):
        self.vivo = False
        print(f"\n [MUNDO] {self.nome} foi abatido!")

    def imobilizar(self):
        self.imobilizado = True
        print(f"\n [MUNDO] {self.nome} está imobilizado!")

In [ ]:
def testar_requisitos(jogador, lista_requisitos):
  for req in lista_requisitos:
    tipo = req["tipo"]

    if tipo == "atributo":
      if not jogador.tem_atributo(req["nome"], req["valor"]):
        return False
    elif tipo == "magia":
      tem_nivel = jogador.tem_atributo(req["nome"], req["valor"])
      tem_mp = jogador.tem_mp(req["custo_mp"])
      if not (tem_nivel and tem_mp):
        return False
    elif tipo == "item":
      if not jogador.tem_item(req["nome"]):
        return False
    elif tipo == "moedas":
      if not jogador.tem_moedas(req["valor"]):
        return False

  return True

In [ ]:
def processar_escolha(jogador, opcao_escolhida):
  if "proxima_cena" in opcao_escolhida:
    return opcao_escolhida["proxima_cena"]

  modos = opcao_escolhida.get("modos", [])
  modo_bem_sucedido = None

  for modo in modos:
    if testar_requisitos(jogador, modo["requisitos"]):
      modo_bem_sucedido = modo
      break
  if modo_bem_sucedido:
    for req in modo_bem_sucedido["requisitos"]:
      if req["tipo"] == "magia":
        jogador.consumir_mp(req["custo_mp"])

    narrativa_formatada = formatar_texto(modo_bem_sucedido["narrativa"])

    print("\n" + "=" * 40)
    print(narrativa_formatada)
    print("=" * 40 + "\n")
# -----SISTEMA DE EQUIPAMENTOS-----
    if "equipar_item" in modo_bem_sucedido:
      item = modo_bem_sucedido["equipar_item"]
      if item["nome"] not in jogador.inventario:
        jogador.adicionar_item(item["nome"])
      jogador.equipar_item(item)

    if "perder_equipamento_slot" in modo_bem_sucedido:
      slot = modo_bem_sucedido["perder_equipamento_slot"]
      item_perdido = jogador.desequipar_item(slot)
      if item_perdido:
        jogador.remover_item(item_perdido["nome"])
        print(f"[PERDA] O item '{item_perdido['nome']}' foi perdido!")

# -----SISTEMA DE ITENS DO INVENTARIO------
    if "item_removido" in modo_bem_sucedido:
      for item in modo_bem_sucedido["item_removido"]:
        jogador.remover_item(item)

    if "item_adquirido" in modo_bem_sucedido:
      for item in modo_bem_sucedido["item_adquirido"]:
        jogador.adicionar_item(item)

# -----SISTEMA DE XP-----
    if "xp_ganha" in modo_bem_sucedido:
      jogador.ganhar_xp(modo_bem_sucedido["xp_ganha"])

# -----SISTEMA DE DANO-----
    if "dano_recebido" in modo_bem_sucedido:
      dano = modo_bem_sucedido["dano_recebido"]
      jogador.receber_dano(dano)

    if jogador.hp_atual <=0:
      print("\n[GAME OVER] Você sucumbiu aos ferimentos...")
      return None
# -----SISTEMA DE NPC-----

    if "novo_npc" in modo_bem_sucedido:
      dados = modo_bem_sucedido["novo_npc"]
      id_npc = dados["id_npc"]
      nova_faccao = dados.get("faccao", "neutro")

      if id_npc in PORTADORES_PRINCIPAIS:
        npc_obj = PORTADORES_PRINCIPAIS[id_npc]
        npc_obj.faccao = nova_faccao
        jogador.registrar_npc(
            id_npc=id_npc,
            npc_objeto=npc_obj
        )
      else:
        jogador.registrar_npc(
          id_npc=id_npc,
          nome=dados["nome"],
          faccao=dados.get("faccao", "neutro")
        )

    if "efeito_npc" in modo_bem_sucedido:
      efeito = modo_bem_sucedido["efeito_npc"]
      npc = jogador.obter_npc(efeito["id_npc"])
      if npc:
        if efeito["acao"] == "imobilizar":
          npc.imobilizar()
        elif efeito["acao"] == "abater":
          npc.abater()
        elif efeito["acao"] == "mudar_faccao":
          npc.faccao = efeito["nova_faccao"]

# -----SISTEMA DE MOEDAS-----

    if "gasto_moedas" in modo_bem_sucedido:
      jogador.gastar_moedas(modo_bem_sucedido["gasto_moedas"])

    if "ganho_moedas" in modo_bem_sucedido:
      base = modo_bem_sucedido["ganho_moedas"]
      if "multiplicador_moedas" in modo_bem_sucedido:
        jogador.adicionar_moedas(base, multiplicador=modo_bem_sucedido["multiplicador_moedas"])
      elif "divisao_moedas" in modo_bem_sucedido:
        jogador.dividir_moedas(base, divisao=modo_bem_sucedido["divisao_moedas"])
      else:
        jogador.adicionar_moedas(base)

    return modo_bem_sucedido["proxima_cena"]

  else:
    print("\n[!] Escolha inválida!" )
    return None

In [ ]:
def exibir_hud(jogador):
  print("=" * 50)
  print(f"Nome: {jogador.nome} | Nível: {jogador.nivel} | Pontos Disponíveis: {jogador.pontos_disponiveis}")
  print(f"HP: {jogador.hp_atual}/{jogador.hp_max} | MP: {jogador.mp_atual}/{jogador.mp_max}")
  print(f"FOR: {jogador.obter_atributo_total('forca')} | DES: {jogador.obter_atributo_total('destreza')}")
  print(f"INT: {jogador.obter_atributo_total('inteligencia')} | SOR: {jogador.obter_atributo_total('sorte')}")
  print(f"MAG: {jogador.obter_atributo_total('magia')} | VIT: {jogador.obter_atributo_total('vitalidade')}")
  print(f"Moedas: {jogador.moedas}C")

  if jogador.inventario:
    itens_str = ", ".join(jogador.inventario)
  else:
    itens_str = "Vazio"
  print(f"Inventário/Bençãos: {itens_str}")

  if jogador.equipamentos:
    print("-"* 50)
    print("Equipamentos Ativos:")
    for slot, item in jogador.equipamentos.items():
      print(f" • [{slot.upper()}]: {item['nome']}")
  print("=" * 50)

In [ ]:
def avaliar_condicao_oculta(jogador, condicao):
    tipo = condicao.get("tipo")

    if tipo == "atributo":
        return jogador.tem_atributo(condicao["nome"], condicao["valor"])

    elif tipo == "item":
        return jogador.tem_item(condicao["nome"])

    elif tipo == "hp_minimo":
        return jogador.hp_atual >= condicao["valor"]

    elif tipo == "mp_minimo":
        return jogador.mp_atual >= condicao["valor"]

    return False

In [ ]:
import textwrap
import os

def obter_diretorio_base():
  caminho_drive = "/content/drive/MyDrive/Colab Notebooks"

  if os.path.exists("/content/drive"):
    return caminho_drive

  if "__file__" in globals():
    return os.path.dirname(os.path.abspath(__file__))
  return os.getcwd()

def carregar_texto(caminho_arquivo, largura=80):
    diretorio_atual = obter_diretorio_base()
    caminho_completo = os.path.join(diretorio_atual, caminho_arquivo)

    try:
        with open(caminho_completo, 'r', encoding='utf-8') as arquivo:
            conteudo = arquivo.read()
            return formatar_texto(conteudo, largura=largura)

    except FileNotFoundError:
        print(f"[!] Arquivo não encontrado: {caminho_completo}")
        return "Texto indisponível no momento."

def formatar_texto(texto, largura=80):
    if not texto:
        return ""

    paragrafos = texto.split('\n\n')
    paragrafos_formatados = []

    for p in paragrafos:
        p_limpo = " ".join(p.split())
        p_formatado = textwrap.fill(p_limpo, width=largura)
        paragrafos_formatados.append(p_formatado)

    return "\n\n".join(paragrafos_formatados)

In [ ]:
def rodar_cena(jogador, cena):
  if "checks_ocultos" in cena:
    for condicao in cena["checks_ocultos"]:
     if avaliar_condicao_oculta(jogador, condicao):
       return condicao["cena_sucesso"]
    return cena["cena_falha"]

  permite_descanso = cena.get("permite_descanso", False)
  jogador.permitir_distribuicao(permite_descanso)
  if permite_descanso:
    jogador.descansar()

  while True:
    exibir_hud(jogador)

    titulo = cena.get("titulo")

    if titulo:
      print("\n" + "=" * 40)
      print(cena["titulo"])
      print("=" * 40)
    else:
      print("\n")

    print("\n" + formatar_texto(cena["narrativa"]))

    if "proxima_cena" in cena and "opcoes" not in cena:
      input("\nPressione Enter para continuar...")
      return cena["proxima_cena"]

    if "opcoes" not in cena or not cena["opcoes"]:
      print("\n[Fim deste capítulo]")
      return None

    print("\nOpções:")
    for letra, dados_opcao in cena["opcoes"].items():
      texto_opcao = formatar_texto(dados_opcao['texto'])
      print(f"[{letra}] {texto_opcao}")

    entrada = input("\nEscolha uma opção: ").strip().upper()
    if entrada not in cena["opcoes"]:
      print("\n[!] Escolha inválida!")
      input("Pressione Enter para tentar novamente...")
      continue

    opcoes_selecionadas = cena["opcoes"][entrada]
    proxima_cena = processar_escolha(jogador, opcoes_selecionadas)

    if proxima_cena is not None:
      return proxima_cena

    input("Pressione Enter para tentar novamente...")

In [ ]:
cenas = {
    "prologo": {
        "titulo": "PRÓLOGO - O JOGO DIVINO",
        "narrativa": carregar_texto("prologo.txt"),
        "proxima_cena": "capitulo_1",
    },

    "capitulo_1": {
      "titulo": "Capítulo 1 – O Primeiro Encontro",
      "narrativa": carregar_texto("capitulo_1.txt"),
      "opcoes": {
            "A": {
                "texto": """Tentar atrair a atenção dos lobos, criando uma brecha para que a garota finalize a conjuração do feitiço. (Inteligência 1)""",
                "modos": [
                    {
                      "requisitos": [
                            {
                                "tipo": "atributo",
                                "nome": "inteligencia",
                                "valor":1
                            }
                        ],
                      "narrativa": """Noto que, próximo aos incensos, há algumas flores que, quando queimadas junto ao odor da erva, geram o efeito oposto: em vez de atrair, afastam os predadores. Chuto os incensos com força na direção do arbusto florido. Os lobos se assustam com o movimento repentino e recuam alguns passos.""",
                      "proxima_cena": "capitulo_1_resgate",
                      "xp_ganha": 5,
                    }
                ]
            },
          "B": {
              "texto": """Colocar-me entre os lobos e a garota, destruindo os incensos na esperança de dispersar a agressividade dos predadores.""",
              "modos": [
                  {
                      "requisitos": [],
                      "narrativa": """Corro na direção da garota, colocando-me entre os lobos e sua presa. Um deles salta no mesmo instante, fazendo com que eu caia ao lado de um dos incensos. Aproveitando o momento de impacto no chão, arremesso o incenso na direção dos animais. Ele cai sobre um arbusto florido, que começa a queimar e exalar uma fumaça densa, deixando os lobos desorientados por um breve momento.""",
                      "dano_recebido": 5,
                      "xp_ganha": 5,
                      "proxima_cena": "capitulo_1_resgate",
                  }
              ]
            },
#          "C": {
#              "texto": """Ignorar o apelo da garota e a notificação do Sistema, virando as costas e indo embora.""",
#              "modos": [
#                  {
#                      "requisitos": [],
#                      "narrativa": """Decido que não vale a pena arriscar minha vida por uma desconhecida. Viro as costas e me preparo para ir embora.""",
#                      "proxima_cena": "capitulo_1_abandono",
#                  }
#              ],
#            },
        }
    },

    "capitulo_1_resgate": {
        "narrativa": carregar_texto("capitulo_1_resgate.txt"),
        "opcoes": {
            "A": {
                "texto": "Apresentar-me e dizer meu nome.",
                "modos": [
                    {
                        "requisitos": [],
                        "narrativa": """— Meu nome é Kael — respondo, ajudando-a a se sentar. — Sou morador de um vilarejo próximo e notei a movimentação incomum na mata. Ainda bem que consegui chegar a tempo.""",
                        "novo_npc": {
                            "id_npc": "campeao_astrid",
                            "nome": "Astrid",
                            "faccao": "aliado",
                        },
                        "proxima_cena": "capitulo_1_lobo",

                    }
                ]
            },
            "B": {
                "texto": "Ignorar a pergunta e focar em sair do local rapidamente.",
                "modos": [
                    {
                        "requisitos": [],
                        "narrativa": """— Meu nome não é importante no momento — digo de forma calma. — Vamos cuidar das suas feridas antes de pensarmos em retornar ao vilarejo.""",
                        "novo_npc": {
                            "id_npc": "campeao_astrid",
                            "nome": "Astrid",
                            "faccao": "aliado"
                        },
                        "proxima_cena": "capitulo_1_lobo",
                    }
                ]
            },
        }
    },

    "capitulo_1_lobo": {
        "narrativa": carregar_texto("capitulo_1_lobo.txt"),
        "opcoes": {
            "A": {
                "texto": "Pedir para que Astrid finalize o lobo e complete a missão extra.",
                "modos":[
                    {
                        "requisitos": [],
                        "narrativa": """Astrid reúne o restante das forças que lhe sobram e dispara mais uma lâmina de vento, cortando a garganta do lobo ferido.
===========================================================
[Missão Concluída!]
Aura está satisfeita, garantindo ao hospedeiro uma
oportunidade de evolução. Escolha com sabedoria!

[Nota Extra de Nox]
Você completou a missão extra. Com isso, tem o direito de
salvar uma vida para manter o equilíbrio sobre a vida perdida.
Use com sabedoria.
===========================================================
RECOMPENSAS:
[+5 XP]
[Pílula de Restauração]
[Bênção de Nox] (Permite sentir quando alguém está próximo
de se encontrar com o Deus da Morte)
=========================================================== """,
                    "item_adquirido": ["Pílula de Restauração", "Bênção de Nox"],
                    "xp_ganha": 5,
                    "proxima_cena": "capitulo_1_casa",
                    }
                ]
            },
            "B": {
                "texto": "Deixar o lobo escapar para preservar a mana da garota até sairmos da floresta.",
                "modos": [
                    {
                        "requisitos": [],
                        "narrativa": """Decido poupar as energias de Astrid. Deixo o animal assustado fugir manco pela vegetação.
===========================================================
[Missão Concluída!]
Aura está satisfeita, garantindo ao hospedeiro uma
oportunidade de evolução. Escolha com sabedoria!
===========================================================
RECOMPENSAS:
[+5 XP]
=========================================================== """,
                        "xp_ganha": 5,
                        "proxima_cena": "capitulo_1_casa",
                    }
                ]
            },
        }
    },

    "capitulo_1_casa": {
        "narrativa": carregar_texto("capitulo_1_casa.txt"),
        "proxima_cena": "cap_1_check_reacao_eskil"
    },

    "cap_1_check_reacao_eskil": {
        "checks_ocultos": [
            {
                "tipo": "hp_minimo",
                "valor": 20,
                "cena_sucesso": "capitulo_1_casa_A"
            }
        ],
        "cena_falha": "capitulo_1_casa_B",
    },

    "capitulo_1_casa_A": {
        "narrativa": """Eskil mostra aos presentes os restos das ervas queimadas, identificando o "repelente" improvisado que criei na batalha. Olho impressionado para o meu pai; ele conseguiu deduzir minha estratégia observando apenas os poucos rastros do combate.""",
        "proxima_cena": "capitulo_1_casa_final",
    },

    "capitulo_1_casa_B": {
        "narrativa": """Eskil mostra as ervas queimadas que formaram o repelente. Um suor frio escorre pela minha testa. Mal sabem eles que tudo não passou de uma grata e desesperada coincidência...""",
        "proxima_cena": "capitulo_1_casa_final",
    },

    "capitulo_1_casa_final": {
        "narrativa": carregar_texto("capitulo_1_casa_final.txt"),
        "checks_ocultos": [
            {
                "tipo": "item",
                "nome": "Bênção de Nox",
                "cena_sucesso": "capitulo_1_casa_final_bencao",
            }
        ],
        "cena_falha": "capitulo_1_casa_final_sem_bencao",
    },

    "capitulo_1_casa_final_bencao": {
        "narrativa": """Enquanto observo Iris puxar Astrid pela mão em direção ao meu quarto, noto uma névoa escura e opaca flutuando ao redor da minha irmã.
A interface translúcida do Sistema surge imediatamente diante dos meus olhos:

===========================================================
[Aviso de Nox]
• Você pode utilizar a [Pílula de Restauração] para curar Iris.
• Instruções: Misture o medicamento na sopa dela durante o
  jantar. A condição crônica será curada após uma noite de sono.
=========================================================== """,
        "opcoes": {
            "A": {
                "texto": "Usar a Pílula na sopa de Iris (-1 Pílula de Restauração)",
                "modos": [
                    {
                        "requisitos": [
                            {
                                "tipo": "item",
                                "nome": "Pílula de Restauração"
                            }
                        ],
                        "narrativa": """Durante o jantar, misturo a Pílula de Restauração na sopa de Iris. Ela come sem perceber, é possível notar a névoa escura ao redor dela se dissipando lentamente.""",
                        "item_removido": ["Pílula de Restauração"],
                        "proxima_cena": "capitulo_1_casa_jantar_bencao",
                    }
                ],
            },

            "B": {
                "texto": "Guardar a Pílula e não fazer nada",
                "modos": [
                    {
                        "requisitos": [],
                        "proxima_cena": "capitulo_1_casa_final_sem_bencao",
                    }
                ],
            },
        },
    },

    "capitulo_1_casa_jantar_bencao": {
        "narrativa": carregar_texto("capitulo_1_casa_jantar_bencao.txt"),
        "opcoes": {
            "A": {
                "texto": "Aceitar a sugestão do sistema e prosseguir a evolução.",
                "modos": [
                    {
                        "proxima_cena": "caminho_heroico",
                    }
                ],
            },
            "B": {
                "texto": "Recusar a sugestão do sistema e não prosseguir a evolução.",
                "modos": [
                    {
                        "proxima_cena": "capitulo_1_casa_final_sem_bencao",
                    }
                ],
            },
        },
    },

    "capitulo_1_casa_final_sem_bencao": {
        "narrativa": "[CAPITULO EM CONSTRUÇÃO]",
        "proxima_cena": "caminho_heroico"
    },
}

In [ ]:
protagonista = Personagem("Kael")
cena_atual_id = "prologo"

while cena_atual_id in cenas:
  cena_objeto = cenas[cena_atual_id]
  cena_atual_id = rodar_cena(protagonista, cena_objeto)
print("\n[Fim do Livro Interativo]")